# 03: Text Preprocessing and Corpus Building

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/African-Native-Oral-LLM/blob/main/notebooks/03_text_preprocessing.ipynb)

Prepare text data for training: transcription workflows, Mandombe script handling, and corpus construction.

## Objectives
- Transcribe audio to text
- Handle Mandombe script encoding
- Normalize and clean text
- Build training corpus

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q openai-whisper regex unidecode pandas

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import re
import json
from pathlib import Path

import pandas as pd
import whisper

# Paths
DATA_ROOT = Path('/content/drive/MyDrive/african_oral_llm_data')
PROCESSED_DIR = DATA_ROOT / 'processed'
TRANSCRIPTS_DIR = DATA_ROOT / 'transcripts'
CORPUS_DIR = DATA_ROOT / 'corpus'

for d in [TRANSCRIPTS_DIR, CORPUS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("✓ Setup complete")

## 2. Automatic Transcription with Whisper

Use OpenAI's Whisper as a starting point for transcription. Note: It doesn't handle tonal languages perfectly - manual review is essential.

In [ ]:
class WhisperTranscriber:
    """Transcribe audio using Whisper (for initial draft)."""
    
    def __init__(self, model_size='base'):
        print(f"Loading Whisper {model_size} model...")
        self.model = whisper.load_model(model_size)
    
    def transcribe(self, audio_path, language=None):
        """
        Transcribe audio file.
        
        Args:
            audio_path: Path to audio file
            language: ISO 639-1 code (e.g., 'sw' for Swahili)
        """
        result = self.model.transcribe(
            audio_path,
            language=language,
            task='transcribe',
            verbose=False
        )
        
        return {
            'text': result['text'],
            'segments': result['segments'],
            'language': result.get('language', 'unknown')
        }
    
    def batch_transcribe(self, audio_paths, language=None):
        """Transcribe multiple files."""
        results = []
        for path in audio_paths:
            try:
                result = self.transcribe(path, language)
                results.append({'path': path, 'status': 'success', **result})
            except Exception as e:
                results.append({'path': path, 'status': 'error', 'error': str(e)})
        return results

# Initialize
# transcriber = WhisperTranscriber('base')  # or 'small', 'medium', 'large'

## 3. Manual Transcription Interface

In [ ]:
import gradio as gr
from IPython.display import Audio, display

class TranscriptionWorkflow:
    """Semi-automated transcription with manual review."""
    
    def __init__(self, transcripts_dir):
        self.transcripts_dir = Path(transcripts_dir)
        self.pending = []
        self.current_index = 0
    
    def load_segments(self, processed_json_path):
        """Load audio segments needing transcription."""
        with open(processed_json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Filter segments without transcripts
        for seg in data['segments']:
            transcript_path = self.transcripts_dir / f"{seg['segment_id']}.txt"
            if not transcript_path.exists():
                self.pending.append(seg)
        
        print(f"Loaded {len(self.pending)} segments pending transcription")
        return self.pending
    
    def save_transcription(self, segment_id, text, tone_marks=None, notes=""):
        """Save transcription with metadata."""
        transcript_data = {
            'segment_id': segment_id,
            'transcription': text,
            'tone_marks': tone_marks or [],
            'transcriber_notes': notes,
            'timestamp': pd.Timestamp.now().isoformat(),
            'reviewed': False
        }
        
        output_path = self.transcripts_dir / f"{segment_id}.json"
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(transcript_data, f, indent=2, ensure_ascii=False)
        
        # Also save plain text
        text_path = self.transcripts_dir / f"{segment_id}.txt"
        with open(text_path, 'w', encoding='utf-8') as f:
            f.write(text)
        
        return output_path

workflow = TranscriptionWorkflow(TRANSCRIPTS_DIR)
print("✓ Transcription workflow ready")

## 4. Text Normalization for African Languages

In [ ]:
class AfricanTextNormalizer:
    """
    Normalize text for African languages.
    Handles diacritics, tone marks, and special characters.
    """
    
    # Tone combining characters
    TONE_MARKS = {
        '\u0301': 'HIGH',      # combining acute
        '\u0300': 'LOW',       # combining grave
        '\u0302': 'FALLING',    # combining circumflex
        '\u030c': 'RISING',     # combining caron
        '\u0304': 'MID',        # combining macron
        '\u0307': 'HIGH_DOT',   # combining dot above
    }
    
    # Nasalization marks
    NASAL_MARKS = ['\u0303', '̃']  # combining tilde
    
    @classmethod
    def normalize(cls, text, language_code='kik'):
        """
        Normalize text while preserving linguistic features.
        
        Args:
            text: Input text
            language_code: ISO 639-3 code
        """
        # NFC normalization (compose characters)
        text = unicodedata.normalize('NFC', text)
        
        # Language-specific normalization
        if language_code == 'kik':
            text = cls._normalize_kikongo(text)
        elif language_code == 'yor':
            text = cls._normalize_yoruba(text)
        elif language_code == 'swa':
            text = cls._normalize_swahili(text)
        
        # Common cleaning
        text = cls._common_cleaning(text)
        
        return text
    
    @staticmethod
    def _normalize_kikongo(text):
        """Kikongo-specific normalization."""
        # Handle prenasalized consonants
        text = re.sub(r'mb', 'mb', text, flags=re.IGNORECASE)
        text = re.sub(r'nd', 'nd', text, flags=re.IGNORECASE)
        text = re.sub(r'ng', 'ng', text, flags=re.IGNORECASE)
        text = re.sub(r'nv', 'nv', text, flags=re.IGNORECASE)
        
        # Ensure consistent vowel representation
        # e.g., handle ɛ/ɔ variants
        text = text.replace('ɛ', 'ε').replace('ɔ', 'o̩')
        
        return text
    
    @staticmethod
    def _normalize_yoruba(text):
        """Yoruba-specific normalization."""
        # Tone mark consistency
        # Ensure tones are properly positioned
        return text
    
    @staticmethod
    def _normalize_swahili(text):
        """Swahili-specific normalization."""
        # Swahili uses standard Latin script
        # Focus on consistent digraphs
        text = re.sub(r'ch', 'ch', text, flags=re.IGNORECASE)
        text = re.sub(r'sh', 'sh', text, flags=re.IGNORECASE)
        return text
    
    @staticmethod
    def _common_cleaning(text):
        """Common text cleaning."""
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text)
        
        # Normalize punctuation spacing
        text = re.sub(r'\s+([.,!?;:])', r'\1', text)
        
        # Remove control characters
        text = re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f\x7f]', '', text)
        
        return text.strip()
    
    @classmethod
    def extract_tone_pattern(cls, text):
        """Extract tone pattern from text with combining marks."""
        tones = []
        for char in text:
            if char in cls.TONE_MARKS:
                tones.append(cls.TONE_MARKS[char])
        return tones

print("✓ Text normalizer loaded")

## 5. Build Training Corpus

In [ ]:
class CorpusBuilder:
    """Build training corpus from transcribed segments."""
    
    def __init__(self, output_dir):
        self.output_dir = Path(output_dir)
        self.documents = []
    
    def add_document(self, text, metadata=None):
        """Add a document to the corpus."""
        self.documents.append({
            'text': text,
            'metadata': metadata or {}
        })
    
    def add_from_transcriptions(self, transcripts_dir, processed_dir):
        """Add all transcribed segments."""
        
        for transcript_file in Path(transcripts_dir).glob('*.json'):
            with open(transcript_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Find corresponding processed data
            recording_id = data['segment_id'].split('_seg')[0]
            processed_file = Path(processed_dir) / f"{recording_id}_processed.json"
            
            metadata = {'segment_id': data['segment_id']}
            if processed_file.exists():
                with open(processed_file, 'r', encoding='utf-8') as f:
                    processed = json.load(f)
                    # Find segment metadata
                    for seg in processed['segments']:
                        if seg['segment_id'] == data['segment_id']:
                            metadata.update({
                                'tone_analysis': seg.get('tone_analysis', {}),
                                'duration': seg.get('duration', 0)
                            })
                            break
            
            self.add_document(data['transcription'], metadata)
        
        print(f"Added {len(self.documents)} documents")
    
    def build_corpus(self, output_filename='corpus.txt', 
                    min_length=10, max_length=10000):
        """
        Build and save the training corpus.
        
        Args:
            min_length: Minimum characters per document
            max_length: Maximum characters per document
        """
        output_path = self.output_dir / output_filename
        
        with open(output_path, 'w', encoding='utf-8') as f:
            for doc in self.documents:
                text = doc['text']
                
                # Filter by length
                if len(text) < min_length or len(text) > max_length:
                    continue
                
                # Write with document separator
                f.write(text + '\n\n')
        
        # Calculate statistics
        total_chars = sum(len(d['text']) for d in self.documents)
        total_docs = len(self.documents)
        
        stats = {
            'total_documents': total_docs,
            'total_characters': total_chars,
            'average_length': total_chars / total_docs if total_docs > 0 else 0,
            'output_file': str(output_path)
        }
        
        # Save stats
        stats_path = self.output_dir / 'corpus_stats.json'
        with open(stats_path, 'w', encoding='utf-8') as f:
            json.dump(stats, f, indent=2)
        
        print(f"✓ Corpus built: {output_path}")
        print(f"  Documents: {stats['total_documents']}")
        print(f"  Characters: {stats['total_characters']:,}")
        print(f"  Avg length: {stats['average_length']:.0f} chars")
        
        return stats

builder = CorpusBuilder(CORPUS_DIR)
print("✓ Corpus builder initialized")

## 6. Create Training Datasets

In [ ]:
def create_datasets_from_corpus(corpus_path, tokenizer, 
                                 train_split=0.9, val_split=0.05, test_split=0.05):
    """
    Split corpus into train/val/test sets.
    Returns HuggingFace datasets.
    """
    from datasets import Dataset
    
    # Read corpus
    with open(corpus_path, 'r', encoding='utf-8') as f:
        text = f.read()
    
    # Split into documents
    documents = [d.strip() for d in text.split('\n\n') if d.strip()]
    
    # Shuffle
    import random
    random.seed(42)
    random.shuffle(documents)
    
    # Calculate split indices
    n = len(documents)
    train_end = int(n * train_split)
    val_end = train_end + int(n * val_split)
    
    train_docs = documents[:train_end]
    val_docs = documents[train_end:val_end]
    test_docs = documents[val_end:]
    
    # Create datasets
    train_dataset = Dataset.from_dict({'text': train_docs})
    val_dataset = Dataset.from_dict({'text': val_docs})
    test_dataset = Dataset.from_dict({'text': test_docs})
    
    print(f"✓ Datasets created:")
    print(f"  Train: {len(train_docs)} documents")
    print(f"  Val: {len(val_docs)} documents")
    print(f"  Test: {len(test_docs)} documents")
    
    return train_dataset, val_dataset, test_dataset

# Example usage when corpus exists:
# corpus_file = CORPUS_DIR / 'corpus.txt'
# if corpus_file.exists():
#     train_ds, val_ds, test_ds = create_datasets_from_corpus(corpus_file, None)

## Summary

This notebook provides:
- **Automated transcription** with Whisper (initial draft)
- **Manual review workflow** for accuracy
- **Text normalization** for African languages
- **Corpus building** from transcripts
- **Dataset creation** for training

Next: Proceed to Notebook 04 for custom tokenizer training.